# Lecture 7 -- Style Token Fine-Tuning

Added a style embedding to GPT so it can switch between corpus generation (style 0) and conversation mode (style 1). Fine-tune our pretrained model on 5k SFT pairs with cosine LR decay + warmup.

In [1]:
import torch
from torch import nn
import torch.nn.functional as F
import numpy as np
import math

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')
if torch.cuda.device_count() > 1:
    print(f'GPUs: {torch.cuda.device_count()} (fine-tuning uses 1)')

device: cuda
GPUs: 2 (fine-tuning uses 1)


## 1. Tokenizer and Datasets

SFT data uses `<EOS>` after each LUFFY response so the model learns when to stop.

In [2]:
# Build tokenizer from the main corpus (so vocab is consistent with pretrained model)
corpus_path = 'dataset/processed/corpus_clean.txt'
sft_path    = 'dataset/luffy_sft.txt'

with open(corpus_path) as f:
    corpus_text = f.read()
with open(sft_path) as f:
    sft_text = f.read()

# Build vocab from corpus (same as pretrained model)
chars = sorted(set(corpus_text))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

def encode(text):
    return [char_to_idx[c] for c in text if c in char_to_idx]

def decode(ids):
    return ''.join(idx_to_char[i] for i in ids)

print(f'Corpus: {len(corpus_text):,} chars')
print(f'SFT:    {len(sft_text):,} chars')
print(f'Vocab:  {vocab_size} unique characters')

Corpus: 3,223,998 chars
SFT:    564,444 chars
Vocab:  96 unique characters


In [3]:
# Check all SFT chars are in vocab (no unknown characters)
unknown = set(sft_text) - set(chars)
if unknown:
    print(f'WARNING: {len(unknown)} chars in SFT not in vocab: {unknown}')
    print('These will be skipped during encoding.')
else:
    print('All SFT characters are in the corpus vocab.')

These will be skipped during encoding.


In [4]:
# Hyperparams — must match pretrained model
context_size = 256
batch_size   = 32    # smaller for fine-tuning
n_embd       = 384
n_head       = 6
n_layer      = 6
n_styles     = 2    # style 0 = corpus, style 1 = SFT

# Encode datasets
corpus_data = torch.tensor(encode(corpus_text), dtype=torch.long)
sft_data    = torch.tensor(encode(sft_text),    dtype=torch.long)

print(f'Corpus tokens: {len(corpus_data):,}')
print(f'SFT tokens:    {len(sft_data):,}')

Corpus tokens: 3,223,998
SFT tokens:    554,169


In [5]:
class Dataset:
    def __init__(self, data, context_size, batch_size, split_factor=0.9):
        self.context_size = context_size
        self.batch_size   = batch_size
        n = int(len(data) * split_factor)
        self.train_data = data[:n]
        self.val_data   = data[n:]

    def get_batch(self, split, device):
        data = self.train_data if split == 'train' else self.val_data
        ix = torch.randint(len(data) - self.context_size - 1, (self.batch_size,))
        x = torch.stack([data[i:i+self.context_size] for i in ix])
        y = torch.stack([data[i+1:i+self.context_size+1] for i in ix])
        return x.to(device), y.to(device)


class MultiStyleDataset:
    """Randomly samples from multiple datasets, returning (x, y, style_index)."""
    def __init__(self, datasets, probs):
        assert abs(sum(probs) - 1.0) < 1e-6
        self.datasets = datasets
        self.probs    = probs

    def get_batch(self, split, device):
        idx = np.random.choice(len(self.datasets), p=self.probs)
        x, y = self.datasets[idx].get_batch(split, device)
        style = torch.full((x.size(0),), idx, dtype=torch.long, device=device)
        return x, y, style


corpus_dataset = Dataset(corpus_data, context_size, batch_size)
sft_dataset    = Dataset(sft_data,    context_size, batch_size)

# 20% corpus (keep language skills) / 80% SFT (learn conversation)
multi_dataset = MultiStyleDataset([corpus_dataset, sft_dataset], [0.2, 0.8])

print('Datasets ready.')

Datasets ready.


## 2. Model with Style Embedding

Same GPT architecture + `style_embedding_table = nn.Embedding(n_styles, n_embd)` prepended at position 0. Also added sampling controls (temperature, top-k, top-p, repetition penalty).

In [6]:
class Head(nn.Module):
    def __init__(self, head_size, n_embd, context_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(0.2)
        self.register_buffer('tril', torch.tril(torch.ones(context_size, context_size)))

    def forward(self, x, kv_cache=None):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=1)
            v = torch.cat([kv_cache[1], v], dim=1)
        new_cache = (k, v)
        T_full = k.shape[1]
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[T_full-T:T_full, :T_full] == 0, float('-inf'))
        wei = self.dropout(F.softmax(wei, dim=-1))
        return wei @ v, new_cache


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, context_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, context_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x, kv_cache=None):
        if kv_cache is None:
            kv_cache = [None] * len(self.heads)
        results = [h(x, c) for h, c in zip(self.heads, kv_cache)]
        outs, new_caches = zip(*results)
        return self.dropout(self.proj(torch.cat(list(outs), dim=-1))), list(new_caches)


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd * 4), nn.ReLU(),
            nn.Linear(n_embd * 4, n_embd), nn.Dropout(0.2),
        )
    def forward(self, x): return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, context_size):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size, n_embd, context_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x, kv_cache=None):
        sa_out, new_cache = self.sa(self.ln1(x), kv_cache)
        x = x + sa_out
        x = x + self.ffwd(self.ln2(x))
        return x, new_cache


def apply_sampling(logits, generated_ids, temperature=1.0, top_k=0, top_p=0.0, repetition_penalty=1.0):
    if logits.dim() == 3:
        logits = logits[:, -1, :]
    if repetition_penalty != 1.0:
        for tok in set(generated_ids[0].tolist()):
            if logits[0, tok] > 0:
                logits[0, tok] /= repetition_penalty
            else:
                logits[0, tok] *= repetition_penalty
    if temperature != 1.0:
        logits = logits / temperature
    if top_k > 0:
        vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < vals[:, -1:]] = float('-inf')
    if top_p > 0.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        mask = cum_probs - F.softmax(sorted_logits, dim=-1) >= top_p
        sorted_logits[mask] = float('-inf')
        logits = sorted_logits.scatter(1, sorted_idx, sorted_logits)
    return logits


class GPTWithStyle(nn.Module):
    def __init__(self, vocab_size, n_embd, context_size, n_head, n_layer, n_styles):
        super().__init__()
        self.context_size = context_size
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(context_size, n_embd)
        self.style_embedding_table    = nn.Embedding(n_styles, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, context_size) for _ in range(n_layer)])
        self.ln_f   = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, style, targets=None, use_cache=False, kv_cache=None):
        if use_cache and kv_cache is not None and kv_cache[0] is not None:
            past_len = kv_cache[0][0][0].shape[1]
            emb = self.token_embedding_table(idx)
            pos = self.position_embedding_table(
                torch.arange(past_len, past_len + idx.shape[1], device=idx.device))
            x = emb + pos
        else:
            suffix = idx[:, -(self.context_size - 1):]
            T = suffix.shape[1] + 1
            style_emb = self.style_embedding_table(style).unsqueeze(1)
            tok_emb   = self.token_embedding_table(suffix)
            emb = torch.cat([style_emb, tok_emb], dim=1)
            pos = self.position_embedding_table(torch.arange(T, device=idx.device))
            x = emb + pos

        if use_cache:
            if kv_cache is None:
                kv_cache = [None] * len(self.blocks)
            new_caches = []
            for block, cache in zip(self.blocks, kv_cache):
                x, nc = block(x, cache)
                new_caches.append(nc)
        else:
            new_caches = None
            for block in self.blocks:
                x, _ = block(x)

        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        if use_cache:
            return logits, loss, new_caches
        return logits, loss

    @torch.no_grad()
    def generate(self, prompt_idx, style, num_tokens=300,
                 temperature=1.0, top_k=0, top_p=0.0, repetition_penalty=1.0,
                 eos_tokens=None):
        self.eval()
        idx = prompt_idx
        prompt_len = prompt_idx.shape[1]
        kv_cache = None
        for _ in range(num_tokens):
            if kv_cache is not None and kv_cache[0][0][0].shape[1] >= self.context_size:
                kv_cache = None
            if kv_cache is not None:
                logits, _, kv_cache = self(idx[:, -1:], style, use_cache=True, kv_cache=kv_cache)
            else:
                idx_in = idx[:, -(self.context_size - 1):]
                logits, _, kv_cache = self(idx_in, style, use_cache=True)
            logits = apply_sampling(logits, idx, temperature, top_k, top_p, repetition_penalty)
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_tok], dim=1)
            # stop at EOS
            if eos_tokens is not None:
                generated = idx[0, prompt_len:].tolist()
                gen_str = ''.join([chr(t) if t < 128 else '' for t in generated])
                if '<EOS>' in gen_str:
                    break
        return idx[:, prompt_len:]

print('GPTWithStyle defined.')

GPTWithStyle defined.


## 3. Load Pretrained Weights

Rename `ln_head` -> `lm_head` to match, load with `strict=False` so `style_embedding_table` inits fresh.

In [7]:
model = GPTWithStyle(vocab_size, n_embd, context_size, n_head, n_layer, n_styles).to(device)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Parameters: {total_params:.2f}M')

# Load pretrained weights
checkpoint = torch.load('luffy_gpt.pth', map_location=device)

# Rename ln_head -> lm_head (our old name vs new name)
if 'ln_head.weight' in checkpoint:
    checkpoint['lm_head.weight'] = checkpoint.pop('ln_head.weight')
    checkpoint['lm_head.bias']   = checkpoint.pop('ln_head.bias')

missing, unexpected = model.load_state_dict(checkpoint, strict=False)
print(f'Missing keys (newly initialised): {missing}')
print(f'Unexpected keys (ignored):        {unexpected}')
print('Pretrained weights loaded!')

Parameters: 10.81M
Missing keys (newly initialised): ['style_embedding_table.weight']
Unexpected keys (ignored):        []
Pretrained weights loaded!


## 4. Fine-Tuning

Uses cosine LR decay with warmup, weight decay, and early stopping at best val loss.

In [8]:
@torch.no_grad()
def estimate_loss(model, datasets, eval_iters=50):
    model.eval()
    results = {}
    for style_idx, dataset in enumerate(datasets):
        for split in ['train', 'val']:
            losses = []
            for _ in range(eval_iters):
                x, y = dataset.get_batch(split, device)
                style = torch.full((x.size(0),), style_idx, dtype=torch.long, device=device)
                _, loss = model(x, style, targets=y)
                losses.append(loss.item())
            results[f'style{style_idx}_{split}'] = np.mean(losses)
    model.train()
    return results


def finetune(model, multi_dataset, steps=5000, lr=5e-5, report_every=500,
             warmup_steps=200, weight_decay=0.1):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # cosine decay with linear warmup
    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / max(1, steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    model.train()

    best_val = float('inf')
    best_state = None

    for step in range(steps):
        x, y, style = multi_dataset.get_batch('train', device)
        _, loss = model(x, style, targets=y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        scheduler.step()

        if step % report_every == 0 or step == steps - 1:
            losses = estimate_loss(model, multi_dataset.datasets)
            cur_lr = scheduler.get_last_lr()[0]
            parts = [f'{k}: {v:.4f}' for k, v in losses.items()]
            print(f'step {step:>5}  lr={cur_lr:.2e}  |  ' + '  '.join(parts))

            # save best checkpoint based on style1 val loss
            if losses['style1_val'] < best_val:
                best_val = losses['style1_val']
                best_state = {k: v.clone() for k, v in model.state_dict().items()}

    # restore best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f'\nrestored best checkpoint (style1_val: {best_val:.4f})')

    return model

print('ready')

ready


In [9]:
# fine-tune with cosine LR + warmup + early stopping
model = finetune(model, multi_dataset, steps=10000, lr=5e-5, report_every=500,
                 warmup_steps=200, weight_decay=0.1)

torch.save(model.state_dict(), 'luffy_gpt_finetuned.pth')
print('saved -> luffy_gpt_finetuned.pth')

step     0  lr=2.50e-07  |  style0_train: 0.7575  style0_val: 1.2387  style1_train: 2.7284  style1_val: 2.7492
step   500  lr=4.99e-05  |  style0_train: 0.8348  style0_val: 1.2265  style1_train: 0.9045  style1_val: 0.9425
step  1000  lr=4.92e-05  |  style0_train: 0.8434  style0_val: 1.2281  style1_train: 0.8173  style1_val: 0.8764
step  1500  lr=4.79e-05  |  style0_train: 0.8411  style0_val: 1.2186  style1_train: 0.7578  style1_val: 0.8477
step  2000  lr=4.59e-05  |  style0_train: 0.8449  style0_val: 1.2194  style1_train: 0.7111  style1_val: 0.8224
step  2500  lr=4.35e-05  |  style0_train: 0.8406  style0_val: 1.2256  style1_train: 0.6730  style1_val: 0.8155
step  3000  lr=4.06e-05  |  style0_train: 0.8474  style0_val: 1.2267  style1_train: 0.6434  style1_val: 0.7956
step  3500  lr=3.73e-05  |  style0_train: 0.8485  style0_val: 1.2313  style1_train: 0.6087  style1_val: 0.7827
step  4000  lr=3.36e-05  |  style0_train: 0.8461  style0_val: 1.2426  style1_train: 0.5999  style1_val: 0.7862
s

## 5. Generation with Sampling Controls

Compare style 0 (corpus) vs style 1 (conversation) with temperature, top-k, top-p, and repetition penalty.

In [10]:
eos_tokens = encode('<EOS>')

def chat(prompt, style_idx, num_tokens=200, temperature=0.8, top_k=50, top_p=0.9, repetition_penalty=1.2):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    style = torch.tensor([style_idx], dtype=torch.long, device=device)
    out = model.generate(idx, style, num_tokens=num_tokens,
                         temperature=temperature, top_k=top_k, top_p=top_p,
                         repetition_penalty=repetition_penalty,
                         eos_tokens=eos_tokens)
    # generate now returns only the generated part (no prompt)
    result = decode(out[0].tolist())
    result = result.split('<EOS>')[0].split('\nUSER:')[0].strip()
    return result

prompt = 'USER: Who are you?\nLUFFY:'

print('=== Style 0 (corpus) ===')
print(chat(prompt, style_idx=0))
print()
print('=== Style 1 (SFT conversation) ===')
print(chat(prompt, style_idx=1))

=== Style 0 (corpus) ===
Mostly! I'm hungry. Not him! EOS

=== Style 1 (SFT conversation) ===
A clown pirate! He's gonna be King of the Pirates! EOS


In [ ]:
prompts = [
    'USER: What is your dream?\nLUFFY:',
    'USER: Are you hungry?\nLUFFY:',
    'USER: Are you scared?\nLUFFY:',
    'USER: Who is your crew?\nLUFFY:',
    'USER: Who are you?\nLUFFY:',
    'USER: Who is Shanks?\nLUFFY:',
]

for p in prompts:
    q = p.split('USER: ')[1].split('\n')[0]
    result = chat(p, style_idx=1, num_tokens=150)
    print(f'You: {q}')
    print(f'Luffy: {result}')
    print()

## 6. Interactive Chat

In [12]:
print('Luffy Chat -- type a message, blank to quit')
print('-' * 40)
while True:
    user_input = input('You: ').strip()
    if not user_input:
        break
    prompt = f'USER: {user_input}\nLUFFY:'
    result = chat(prompt, style_idx=1, num_tokens=120)
    print(f'Luffy: {result}')
    print()

Luffy Chat -- type a message, blank to quit
----------------------------------------
Luffy: He'd do what feels right. And she hits me when I do dumb stuff. EOS

